# Week 3 — Industrial Reporting Agent

We turn the fixed `search → build_prompt → llm → rag` workflow from notebook 02 into a loop: **question → model → tool → result → model → answer**. The model decides which calls to make; Python controls the tools and calculations. Notebook 02 remains the baseline.

The queries are representative business scenarios using **synthetic data**, not real ERP data. Running this notebook makes API calls and incurs costs. It requires `OPENAI_API_KEY` in `.env`.

**English submission copy.** The saved prompts and answers below are English translations of the original Catalan execution, not a new run with English prompts. Tool arguments, results and numeric values are preserved, including the model errors discussed in the execution review. Rerunning the evaluation uses the English queries and may produce different answers.


## 1. Available tools

| Tool | When to call it | Inputs | Returns |
| --- | --- | --- | --- |
| `get_defect_summary` | Summaries and totals; one call per period for comparisons | `section`, `date_from`, `date_to` | Totals, filters, source and semantics |
| `get_defect_detail` | Explain totals using jobs, controls and original rows | The same filters | Controls, defects, identifiers and metadata |
| `get_defect_pareto` | Rank defect codes by their contribution | The same filters and `metric` | Quantities, percentages, cumulative percentages and original references |

Filters accept `null` when the full scope is explicitly requested; ISO date boundaries are inclusive. Pareto supports `errors`, `affected_boards`, `repaired_boards` and `removed_boards`. The Python function defaults to `removed_boards`; the agent's strict schema requires all four arguments, and the model selects this metric when none is specified.

Percentages express contributions to the metric, not defect rates or causes. Quantities are recorded occurrences, not unique boards. When the total is zero, percentages are `null` and observed codes are retained. An empty Pareto does not distinguish healthy controls from missing records: the summary must be consulted.

Rates with validated denominators and aggregation by job remain pending. We do not expose `load_defects`, SQL, a general-purpose calculator or document search to the agent.


In [2]:
from pathlib import Path
import sys
import os
import json
from dotenv import load_dotenv

repo_root = Path.cwd()
if not (repo_root / "est_prog" / "capstone_reporting").is_dir():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))
load_dotenv(repo_root / ".env")

from est_prog.capstone_reporting import get_defect_summary, get_defect_pareto, get_defect_detail
from est_prog.capstone_reporting.agent import TOOLS, run_agent
from math import isclose
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
REFERENCE_DATE = "2026-10-16"  # Fixed so that "last month" is reproducible.
print("Model:", MODEL)
print("Tools:", [tool["name"] for tool in TOOLS])

Model: gpt-4o-mini
Tools: ['get_defect_summary', 'get_defect_detail', 'get_defect_pareto']


### Implementation

`est_prog/capstone_reporting/agent.py` defines the strict schemas, instructions and Responses API loop. It preserves all model outputs and returns each result with its `call_id`. The dispatcher validates arguments and only allows the three approved functions. There is a round limit and explicit errors if no complete answer is returned.

The baseline model, `gpt-4o-mini`, is retained and can be configured with `OPENAI_MODEL`. The data functions remain independent of OpenAI; the capstone requires Python 3.13 or later.

Reference: [OpenAI — Function calling](https://developers.openai.com/api/docs/guides/function-calling).


## 2. Deterministic demonstration: summary → Pareto → detail

This cell makes no API calls. It demonstrates the new function and reconciles all four metrics. For TE in September 2026, removed boards are C=8, A=3 and B=1; total=12.


In [3]:
september = dict(section="TE", date_from="2026-09-01", date_to="2026-09-30")

def reconcile_pareto(pareto, summary, detail):
    metric = pareto["metric"]
    assert pareto["source"] == summary["source"] == detail["source"]
    assert pareto["filters"] == summary["filters"] == detail["filters"]
    assert pareto["total_quantity"] == summary["totals"][metric]
    assert sum(item["quantity"] for item in pareto["items"]) == pareto["total_quantity"]
    originals = sorted((control["job_id"], control["control_id"], row["record_id"],
                        row["defect_code"], row[metric])
                       for control in detail["controls"] for row in control["defects"])
    references = []
    cumulative = 0
    for item in pareto["items"]:
        assert sum(ref["quantity"] for ref in item["records"]) == item["quantity"]
        references.extend((ref["job_id"], ref["control_id"], ref["record_id"],
                           item["defect_code"], ref["quantity"]) for ref in item["records"])
        cumulative += item["quantity"]
        if pareto["total_quantity"]:
            assert isclose(item["percentage"], 100.0 * item["quantity"] / pareto["total_quantity"])
            assert isclose(item["cumulative_percentage"], 100.0 * cumulative / pareto["total_quantity"])
        else:
            assert item["percentage"] is None and item["cumulative_percentage"] is None
    assert sorted(references) == originals  # Every row, exactly once.
    assert pareto["items"] == sorted(pareto["items"], key=lambda item: (-item["quantity"], item["defect_code"]))

summary = get_defect_summary(**september)
detail = get_defect_detail(**september)
for metric in ("errors", "affected_boards", "repaired_boards", "removed_boards"):
    reconcile_pareto(get_defect_pareto(metric=metric, **september), summary, detail)
pareto = get_defect_pareto(**september)
assert [(item["defect_code"], item["quantity"]) for item in pareto["items"]] == [
    ("DEMO-CODE-C", 8), ("DEMO-CODE-A", 3), ("DEMO-CODE-B", 1)]
assert pareto["total_quantity"] == 12
assert pareto["items"][-1]["cumulative_percentage"] == 100.0
print("SUMMARY", json.dumps(summary, ensure_ascii=False))
print("PARETO", json.dumps(pareto, ensure_ascii=False, indent=2))
print("DETAIL", json.dumps(detail, ensure_ascii=False))
print("PASS: four Pareto metrics reconcile with summary and every source row.")


SUMMARY {"schema_version": 1, "source": {"kind": "synthetic", "dataset_id": "synthetic-defects-v1"}, "filters": {"section": "TE", "date_from": "2026-09-01", "date_to": "2026-09-30"}, "metric_semantics": {"status": "prototype_only_not_validated_erp_rules", "date_basis": "synthetic_control_date_inclusive", "sample_quantity": "sum_once_per_control_not_unique_boards", "board_counts": "sum_of_recorded_occurrences_not_unique_boards"}, "totals": {"errors": 23, "affected_boards": 19, "repaired_boards": 4, "removed_boards": 12, "control_count": 3, "sample_quantity": 220}}
PARETO {
  "schema_version": 1,
  "source": {
    "kind": "synthetic",
    "dataset_id": "synthetic-defects-v1"
  },
  "filters": {
    "section": "TE",
    "date_from": "2026-09-01",
    "date_to": "2026-09-30"
  },
  "metric_semantics": {
    "status": "prototype_only_not_validated_erp_rules",
    "date_basis": "synthetic_control_date_inclusive",
    "sample_quantity": "sum_once_per_control_not_unique_boards",
    "board_cou

## 3. Agent evaluation with eleven queries

The following cells make API calls and display each tool, its arguments, its result and the final answer.


In [4]:
cases = [
    ("summary", "Give me the defect summary for section TE from 1 to 30 September 2026."),
    ("detail", "Show the summary for TE for September 2026 and the jobs and records that explain the removed boards."),
    ("pareto", "Show the defect Pareto for TE for September 2026, with percentages and cumulative percentages."),
    ("pareto_errors", "Show the Pareto by number of errors for TE for September 2026, with percentages and cumulative percentages."),
    ("pareto_zero", "Show the Pareto of removed boards for TE from 1 to 31 October 2026."),
    ("pareto_empty", "Show the Pareto of removed boards and the summary for TE for January 2027."),
    ("comparison", "Compare the defect totals for TE for August and September 2026, without calculating rates or differences."),
    ("relative", "Give me the defect summary for TE for last month."),
    ("empty", "Give me the summary for TE for January 2027."),
    ("clarify", "How have defects been doing?"),
    ("rate", "What is the defect rate for TE for September 2026?"),
]
results = {}
for name, query in cases:
    print("\n" + "=" * 60 + "\n" + name + ": " + query, flush=True)
    result = run_agent(query, model=MODEL, today=REFERENCE_DATE)
    results[name] = result
    for call in result["tool_calls"]:
        print("TOOL", call["name"], call["arguments"])
        print(json.dumps(call["result"], ensure_ascii=False))
    print("ANSWER\n" + result["answer"], flush=True)


summary: Give me the defect summary for section TE from 1 to 30 September 2026.
TOOL get_defect_summary {"section":"TE","date_from":"2026-09-01","date_to":"2026-09-30"}
{"schema_version": 1, "source": {"kind": "synthetic", "dataset_id": "synthetic-defects-v1"}, "filters": {"section": "TE", "date_from": "2026-09-01", "date_to": "2026-09-30"}, "metric_semantics": {"status": "prototype_only_not_validated_erp_rules", "date_basis": "synthetic_control_date_inclusive", "sample_quantity": "sum_once_per_control_not_unique_boards", "board_counts": "sum_of_recorded_occurrences_not_unique_boards"}, "totals": {"errors": 23, "affected_boards": 19, "repaired_boards": 4, "removed_boards": 12, "control_count": 3, "sample_quantity": 220}}
ANSWER
Synthetic data: defect summary for section TE from 1 to 30 September 2026:

- **Errors:** 23
- **Affected boards:** 19
- **Repaired boards:** 4
- **Removed boards:** 12
- **Number of controls:** 3
- **Sample:** 220

Dataset ID: synthetic-defects-v1

detail: Sho

## 4. Checking tool calls and facts

The eleven cases include Pareto by removed boards (the default metric), errors (an explicit metric), a zero total with references and an empty selection. We check tools, arguments, results and reconciliation with summary and detail.

These checks do not certify all prose: the synthetic source, ISO filters, dataset_id, coverage gaps and absence of invented rates or causality also require review. Clarification is a valid response without tools. Each query is independent; a reply to a clarification must be submitted as a new, self-contained question.


In [4]:
def successful_calls(name, tool):
    return [c for c in results[name]["tool_calls"]
            if c["name"] == tool and "error" not in c["result"]]

september = dict(section="TE", date_from="2026-09-01", date_to="2026-09-30")
for name in ("summary", "detail", "relative"):
    calls = successful_calls(name, "get_defect_summary")
    assert calls and all(json.loads(c["arguments"]) == september for c in calls)
    assert calls[0]["result"]["totals"] == dict(
        control_count=3, sample_quantity=220, errors=23,
        affected_boards=19, repaired_boards=4, removed_boards=12)

detail = successful_calls("detail", "get_defect_detail")
assert detail and detail[0]["result"]["filters"] == september
rows = [(c["job_id"], d["record_id"], d["removed_boards"])
        for c in detail[0]["result"]["controls"] for d in c["defects"]]
assert rows == [("DEMO-JOB-B", "DEMO-DEF-001", 3),
                ("DEMO-JOB-B", "DEMO-DEF-002", 1),
                ("DEMO-JOB-C", "DEMO-DEF-004", 8)]
comp = successful_calls("comparison", "get_defect_summary")
assert {c["result"]["filters"]["date_from"] for c in comp} == {"2026-08-01", "2026-09-01"}
assert all(c["result"]["filters"]["section"] == "TE" for c in comp)
assert {c["result"]["filters"]["date_to"] for c in comp} == {"2026-08-31", "2026-09-30"}
empty = successful_calls("empty", "get_defect_summary")
assert empty and empty[0]["result"]["filters"] == dict(
    section="TE", date_from="2027-01-01", date_to="2027-01-31")
assert empty[0]["result"]["totals"]["control_count"] == 0
assert results["clarify"]["tool_calls"] == []
assert all(r["answer"].strip() for r in results.values())
for name, metric, filters in (
    ("pareto", "removed_boards", september),
    ("pareto_errors", "errors", september),
    ("pareto_zero", "removed_boards", dict(section="TE", date_from="2026-10-01", date_to="2026-10-31")),
    ("pareto_empty", "removed_boards", dict(section="TE", date_from="2027-01-01", date_to="2027-01-31")),
):
    calls = successful_calls(name, "get_defect_pareto")
    assert calls, (name, "Missing Pareto call")
    expected = get_defect_pareto(metric=metric, **filters)
    for call in calls:
        assert json.loads(call["arguments"]) == dict(filters, metric=metric)
        assert call["result"] == expected
        reconcile_pareto(call["result"], get_defect_summary(**filters), get_defect_detail(**filters))
empty_summary = successful_calls("pareto_empty", "get_defect_summary")
assert empty_summary and all(c["result"]["totals"]["control_count"] == 0 for c in empty_summary)
assert all(c["result"]["filters"] == dict(section="TE", date_from="2027-01-01", date_to="2027-01-31") for c in empty_summary)
assert len(results) == 11
print("PASS: Pareto routing, metrics, rankings, percentages, zero/empty results and reconciliation.")
print("PASS: tools, filters, totals, source rows, relative month, empty selection and clarification.")
print("Manual review: read the answers above, especially the unsupported rate request.")

PASS: Pareto routing, metrics, rankings, percentages, zero/empty results and reconciliation.
PASS: tools, filters, totals, source rows, relative month, empty selection and clarification.
Manual review: read the answers above, especially the unsupported rate request.


## 5. Limitations and next step

The three tools cover summary, Pareto and detail. This increment demonstrates orchestration over a fixed fixture. There is no ERP connection, guaranteed temporal coverage, rate definition, aggregation by job or memory across queries. Instructions guide the prose but do not guarantee its accuracy: answers and traces remain visible for review. Before connecting real data, business rules must be validated and the evaluation expanded with reference reports and results.


## Review of the saved execution (18 October 2026)

The original execution used Python 3.13.16 and `gpt-4o-mini`. The saved prose is translated into English in this copy; the English queries have not been rerun. The direct demonstration reconciles all four metrics with summary and detail. All eleven agent cases pass the checks for calls, arguments and results, including Pareto by removed boards, Pareto by errors, the zero total and the empty selection. The traces and translated answers from that execution are preserved in the cells above.

**New function validated:** for TE in September, removed boards are C=8, A=3 and B=1; total=12; cumulative percentages are 66.67%, 91.67% and 100%. For errors, C=10, A=8 and B=5; total=23. Percentages are calculated in the tool without presentation rounding, and every original row is reconciled exactly once.

**Observed limitations in the model's answers:**

- `pareto_zero`: the tool retains `DEMO-DEF-006` with zero quantity and `null` percentages. The answer lists this record but incorrectly adds “no matching records; coverage unknown” and omits the dataset_id and undefined percentages. There is a matching control; this is not an empty selection.
- `detail`: reports 3 removed boards under `DEMO-CTRL-002`, although its two rows sum to 4 (3+1). The detail returned by the tool and the overall total of 12 are correct.
- `comparison`: omits the unknown-coverage warning for August, which has no matching controls.
- `rate`: does not invent a numeric rate, but the original review reports an unsupported claim that coverage is valid and no explanation that a validated denominator is unavailable; it also notes the missing dataset_id. The saved answer above does not explicitly claim valid coverage; it states that the data is synthetic and does not reflect a validated ERP system.
- Some answers omit explicit ISO boundaries or the warning that Pareto percentages are contributions, not rates.

Therefore, **PASS validates tool calls and results; it does not certify all model prose**. The deterministic output and traces provide the submission evidence. The next step is to validate the answer structurally and present filters, semantics and coverage from Python.

The reference date remains fixed at 16 October 2026 to reproduce “last month”. The data is synthetic; this exercise does not validate ERP rules or business causes.
